# MNIST với bốn mô hình


In [1]:
from IPython.display import display
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
from mnist_project.data import (
    load_mnist_npz,
    stratified_train_validation,
    stratified_limit,
)
from mnist_project.models import (
    MultinomialNBScratch,
    SoftmaxRegressionScratch,
    MLPScratch,
    RandomForestModel,
)
from mnist_project.metrics import classification_scores


Metrics unit test: OK


## Dữ liệu
Giữ test chính thức riêng. Pixel không âm được chia 255; chỉ số ảnh và nhãn được shuffle cùng nhau.


In [2]:
# Nạp đúng chỉ số chia tập của lần chạy FULL đã lưu.
X_train_raw, y_train_all, X_test_raw, y_test_all = load_mnist_npz(
    ROOT / "data/raw/mnist.npz"
)
with np.load(ROOT / "data/processed/full_indices.npz") as split_indices:
    train_idx = split_indices["train"]
    val_idx = split_indices["validation"]
X_train = X_train_raw[train_idx].reshape(-1, 784).astype("float32") / 255
X_val = X_train_raw[val_idx].reshape(-1, 784).astype("float32") / 255
print(X_train.shape, X_val.shape)


(54000, 784) (6000, 784)


## Multinomial NB
N_cj = tổng cường độ feature j trong lớp c. theta_cj = (N_cj + alpha)/(tổng_j N_cj + alpha*d). Điểm lớp = log prior + tổng_j x_j log theta_cj. Alpha được chọn bằng Macro F1 validation và lưu trong config.json; pixel là lượng không âm, đây là cách dùng gần đúng của mô hình đa thức.


## Softmax và MLP
Softmax dùng cross entropy và mini-batch GD. MLP thêm các lớp ReLU và backprop. Xem docstring trong models.py.


## Random Forest
RandomForestModel sử dụng RandomForestClassifier của scikit-learn: bootstrap, chọn đặc trưng ngẫu nhiên tại mỗi nút và giảm độ hỗn tạp Gini. Dự đoán bằng xác suất trung bình của các cây. Cấu hình FULL: 40 cây, độ sâu tối đa 14, 28 đặc trưng mỗi nút, lá tối thiểu 2 mẫu.


In [3]:
# Nạp cùng bộ mô hình FULL; không huấn luyện riêng trong notebook.
import pickle
with (ROOT / "result/full/models.pkl").open("rb") as stream:
    models = pickle.load(stream)
print("Các mô hình đã lưu:", list(models))


Các mô hình đã lưu: ['MultinomialNB', 'Softmax LR', 'MLP', 'Random Forest']


Notebook dùng dữ liệu, chỉ số chia tập và mô hình FULL trong `result/full`. Chạy `python -m mnist_project.experiment --mode full --baseline` trước khi cập nhật notebook.


In [4]:
import pandas as pd

display(pd.read_csv(ROOT / "result/full/metrics.csv"))


,model,split,n,accuracy,precision_macro,recall_macro,f1_macro,train_seconds,predict_seconds,predict_ms_per_image
0,MultinomialNB,train,54000,0.824315,0.831219,0.821255,0.822978,0.428348,0.219758,0.004070
1,MultinomialNB,validation,6000,0.824333,0.833049,0.821476,0.823298,0.428348,0.021047,0.003508
2,MultinomialNB,test,10000,0.835600,0.842865,0.832472,0.834122,0.428348,0.037413,0.003741
3,Softmax LR,train,54000,0.930630,0.929844,0.929771,0.929720,3.537452,0.031298,0.000580
4,Softmax LR,validation,6000,0.923167,0.922520,0.922482,0.922382,3.537452,0.003516,0.000586
5,Softmax LR,test,10000,0.922900,0.921916,0.921952,0.921783,3.537452,0.005298,0.000530
6,MLP,train,54000,0.999815,0.999819,0.999821,0.999820,10.387970,0.073123,0.001354
7,MLP,validation,6000,0.979167,0.979094,0.978934,0.978991,10.387970,0.007870,0.001312
8,MLP,test,10000,0.978100,0.978016,0.977895,0.977935,10.387970,0.013624,0.001362
9,Random Forest,train,54000,0.993167,0.993275,0.993158,0.993212,1.902129,0.180419,0.003341
